# 09 – Karbantartási stratégiák és kockázatalapú szemlélet

**10. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- corrective/preventive/CBM/PdM/RBM elkülönítése
- TBM és UBM rövidítések helyes használata
- állapotindikátor és egyszerű RUL becslés
- expected-cost alapú RBM döntés

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Karbantartási érettségi ív

- Corrective: hiba után.
- Preventive TBM: időalapon (**Time-Based Maintenance**).
- Preventive UBM: használatalapon (**Usage-Based Maintenance**).
- CBM: mért állapot alapján.
- PdM: előrejelzett állapot/RUL alapján.
- RBM: kockázat, következmény, költség és opportunity együtt.

A stratégiák nem feltétlenül ‘jobbak’ egymásnál; a megfelelő információ és döntési cél határozza meg a választást.


## 2. Preventive interval – első trade-off

Állandó $\lambda$ esetén $F(T)=1-e^{-\lambda T}$. Egyszerű oktatási költség: $C(T)=C_{PM}+F(T)C_F$. Ez még nem teljes renewal-reward optimalizáció.


In [ ]:
lam=1/3000; C_PM=150_000; C_F=2_000_000
T=np.linspace(50,5000,200); F=1-np.exp(-lam*T); C=C_PM+F*C_F
plt.plot(T,C); plt.xlabel('preventive interval [h]'); plt.ylabel('simplified cost [HUF/cycle]'); plt.grid(alpha=.3); plt.show()


## 3. CBM: indikátor és threshold

Szintetikus sensor-bias indikátort generálunk. Threshold crossing esetén condition-based action indítható.


In [ ]:
rng=np.random.default_rng(4); t=np.arange(120); bias=np.maximum(0,.04*(t-40))+rng.normal(0,.15,len(t)); threshold=2.
alarm=bias>=threshold
plt.plot(t,bias); plt.axhline(threshold,ls='--'); plt.xlabel('time'); plt.ylabel('bias indicator'); plt.grid(alpha=.3); plt.show()
print('first crossing:',t[np.argmax(alarm)] if alarm.any() else None)


## 4. PdM: egyszerű RUL gondolat

Az utolsó 30 ponton lineáris trendet illesztünk és becsüljük a threshold crossing idejét. Ez tanulópélda; valódi prognosztikában uncertainty, drift, covariates és model diagnostics szükséges.


In [ ]:
w=30; slope,intercept=np.polyfit(t[-w:],bias[-w:],1); crossing=(threshold-intercept)/slope if slope>0 else np.inf; RUL=crossing-t[-1]
print('predicted crossing',crossing,'RUL',RUL)


## 5. RBM: maintain now vs wait

Egyszerű döntési költség:

$$C_{expected}=C_{maintenance}+C_{production}+P(H)C_{consequence}.$$

Két opciót hasonlítunk, de hangsúly: safety constraint/ALARP/standard minimum requirement nem monetizálható egyszerűen ki.


In [ ]:
def expected(Cm,Cp,p,Cc): return Cm+Cp+p*Cc
opts=pd.DataFrame([
 {'option':'Maintain now','cost':expected(250_000,800_000,.001,50_000_000)},
 {'option':'Wait for opportunity','cost':expected(250_000,100_000,.02,50_000_000)}])
opts


## 6. Break-even probability

Mekkora várakozási hazard probability mellett azonos a két expected cost?


In [ ]:
Cc=50_000_000; p_now=.001; prod_now=800_000; prod_window=100_000
p_break=(prod_now-prod_window+p_now*Cc)/Cc
print('break-even p_wait=',p_break)


In [ ]:
from safetycourse.maintenance import compare_maintenance_options
compare_maintenance_options([
 {'name':'Maintain now','maintenance_cost':250_000,'production_loss':800_000,'hazard_probability':.001,'consequence_cost':50_000_000},
 {'name':'Wait','maintenance_cost':250_000,'production_loss':100_000,'hazard_probability':.02,'consequence_cost':50_000_000}])


## Próbáld ki!

1. Duplázd a consequence costot és számíts új break-even értéket.
2. Csökkentsd a production loss now értékét 300 000 HUF-ra.
3. Írd le, milyen adat kellene ahhoz, hogy p_wait ne kézi input legyen.
4. Hova sorolnád a proof-test alapú karbantartást, és miért nem mindig egyetlen címke elég?


## Összefoglalás

A karbantartási stratégia a rendelkezésre álló információtól és döntési céltól függ. A következő tanegységben a **szenzoradatból diagnosztikai evidenciát** képezünk, amely később dinamikus kockázati input lesz.
